# Verifica output — SIOPE entrate 2025
Fallback GCS se parquet non presenti in out/data/. Fonte: gs://dataciviclab-clean e gs://dataciviclab-mart.

In [1]:
import duckdb, pandas as pd
from pathlib import Path
con = duckdb.connect()
con.execute('install httpfs; load httpfs')
ROOT = Path('../../out/data')
A = 2025
def local_or_gcs(local_rel, gcs_url):
    p = ROOT / local_rel
    return str(p) if p.exists() else gcs_url

CLEAN = local_or_gcs(
    f'clean/siope_entrate_comuni/{A}/siope_entrate_comuni_{A}_clean.parquet',
    f'https://storage.googleapis.com/dataciviclab-clean/siope/siope_entrate_comuni/{A}/siope_entrate_comuni_{A}_clean.parquet')
MART_PRO = local_or_gcs(
    f'mart/siope_entrate_comuni/{A}/siope_entrate_comuni_agg_labeled.parquet',
    f'https://storage.googleapis.com/dataciviclab-mart/siope/siope_entrate_comuni/{A}/siope_entrate_comuni_agg_labeled.parquet')
print(f'Clean: {CLEAN}')
print(f'Mart:  {MART_PRO}')

Clean: ../../out/data/clean/siope_entrate_comuni/2025/siope_entrate_comuni_2025_clean.parquet
Mart:  ../../out/data/mart/siope_entrate_comuni/2025/siope_entrate_comuni_agg_labeled.parquet


In [2]:
c1 = con.execute(f"""select count(*),count(distinct codice_ente),count(distinct periodo),
  count(*) filter(where importo<0), count(*) filter(where codice_ente is null)
  from read_parquet('{CLEAN}')""").fetchone()
print(f'Righe: {c1[0]:,}\nEnti: {c1[1]:,}\nPeriodi: {c1[2]}\nImporti neg: {c1[3]}\nNull ente: {c1[4]}')

Righe: 4,288,627
Enti: 11,764
Periodi: 12
Importi neg: 3
Null ente: 0


In [3]:
j = con.execute(f"""select round(100.*count(*) filter(where provincia is not null)/count(*),2),
  round(100.*count(*) filter(where has_codgest_match)/count(*),2),
  round(100.*count(*) filter(where denominazione_ente is not null)/count(*),2)
  from read_parquet('{CLEAN}')""").fetchone()
print(f'Territorio join: {j[0]}%\nCodgest match:   {j[1]}%\nEnti trovati:    {j[2]}%')

Territorio join: 100.0%
Codgest match:   99.99%
Enti trovati:    100.0%


In [4]:
display(con.execute(f"""select codice_comparto,count(*)righe,round(sum(importo)/1e8,2)tot
  from read_parquet('{CLEAN}') where codice_comparto is not null
  group by codice_comparto order by codice_comparto""").fetchdf())

,codice_comparto,righe,tot
0,AAI,1273,620.91
1,ASP,4583,1877.96
2,CDC,14314,1436.85
3,EGP,5630,248.47
4,EPF,11982,158.72
5,FLS,1780,569.33
6,MON,112483,13377.50
7,PRO,3849413,132157.17
8,REG,40269,267472.67
9,RIC,8403,7718.79


In [5]:
m = con.execute(f"""select count(*),round(sum(importo_totale_eur),0),
  count(*) filter(where importo_totale_eur<0),
  count(*) filter(where macro_categoria_v2 is null),
  round(100.*count(*) filter(where macro_categoria_v2='Altro')/count(*),2)
  from read_parquet('{MART_PRO}')""").fetchone()
print(f'PRO: {m[0]:,} righe, tot {m[1]/1e9:.2f} mld, neg {m[2]}, null_class {m[3]}, altro {m[4]}%')

PRO: 373,386 righe, tot 115.93 mld, neg 1, null_class 0, altro 66.88%


In [6]:
for s,l in [('regioni','REG'),('sanita','SAN'),('universita','UNI')]:
  p = local_or_gcs(f'mart/siope_entrate_comuni/{A}/siope_entrate_{s}_agg_labeled.parquet',
    f'https://storage.googleapis.com/dataciviclab-mart/siope/siope_entrate_comuni/{A}/siope_entrate_{s}_agg_labeled.parquet')
  r = con.execute(f'select count(*),round(sum(importo_totale_eur)/1e9,2) from read_parquet("{p}")').fetchone()
  print(f'  {l}: {r[0]:>6} righe, {r[1]:>8} mld')

  REG:   4023 righe,   267.47 mld
  SAN:   6447 righe,   175.33 mld
  UNI:   4470 righe,    26.77 mld


In [7]:
display(con.execute(f"""select macro_categoria_v2,descrizione_codice,
  round(sum(importo_totale_eur)/1e9,2)mld from read_parquet('{MART_PRO}')
  where is_titolo_9=false group by all order by mld desc limit 10""").fetchdf())

,macro_categoria_v2,descrizione_codice,mld
0,Imposte proprie,Imposta municipale propria riscossa a seguito ...,14.44
1,Contributi agli investimenti,Contributi agli investimenti da Ministeri,8.08
2,Trasferimenti correnti,Trasferimenti correnti da Regioni e province a...,6.97
3,Fondi perequativi,Fondi perequativi dallo Stato,6.92
4,Imposte proprie,Tassa smaltimento rifiuti solidi urbani riscos...,6.80
5,Imposte proprie,Addizionale comunale IRPEF riscossa a seguito ...,6.70
6,Trasferimenti correnti,Trasferimenti correnti da Ministeri,5.67
7,Contributi agli investimenti,Contributi agli investimenti da Regioni e prov...,4.38
8,Imposte proprie,Tributo comunale sui rifiuti e sui servizi,2.90
9,Altro,Anticipazioni da istituto tesoriere/cassiere,2.85


In [8]:
checks=[
  ('Clean righe >1M', '✅' if c1[0]>1_000_000 else '❌'),
  ('Clean 12 periodi', '✅' if c1[2]==12 else '❌'),
  ('Importi neg <100', '✅' if c1[3]<100 else '⚠️'),
  ('Join territorio >95%', '✅' if j[0]>95 else '❌'),
  ('Join codgest >95%', '✅' if j[1]>95 else '❌'),
  ('Mart PRO righe >100K', '✅' if m[0]>100_000 else '❌'),
]
display(pd.DataFrame(checks,columns=['check','esito']))
con.close()
print('✅ Verifica entrate completata')

,check,esito
0,Clean righe >1M,✅
1,Clean 12 periodi,✅
2,Importi neg <100,✅
3,Join territorio >95%,✅
4,Join codgest >95%,✅
5,Mart PRO righe >100K,✅


✅ Verifica entrate completata
